# Training Loop


Use this notebook after the training loop note. The goal is to turn the abstract loop into a small run with loss, validation, and artifacts you can inspect.

You will:
- run forward, backward, and optimizer steps
- evaluate during the run
- write a checkpoint and a machine-readable report


In [1]:
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from course_tools import CharTokenizer, TinyConfig, TinyTransformerLM, default_artifact_dir, evaluate_model, save_checkpoint, train_model, write_json

LECTURE_NOTE_TITLE = 'Training Loop'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: Training Loop


## Forward backward step repeat


In [2]:
text = 'training loops turn data and gradients into learned weights.\n' * 8
tokenizer = CharTokenizer.build([text])
model = TinyTransformerLM(TinyConfig(vocab_size=len(tokenizer.stoi), block_size=32, d_model=32, n_heads=4, n_layers=1, d_ff=64))
history = train_model(model, tokenizer, train_text=text, eval_text=text, steps=10, learning_rate=3e-3, batch_size=4)
print(history)


[{'step': 1.0, 'train_loss': 21.840639114379883, 'eval_loss': 20.9647159576416, 'bpb': 30.245691745735254}, {'step': 2.0, 'train_loss': 21.454755783081055, 'eval_loss': 20.209707260131836, 'bpb': 29.15644444200988}, {'step': 4.0, 'train_loss': 20.176244735717773, 'eval_loss': 18.489925384521484, 'bpb': 26.675323658656108}, {'step': 6.0, 'train_loss': 18.77252769470215, 'eval_loss': 17.726938247680664, 'bpb': 25.574565900073786}, {'step': 8.0, 'train_loss': 17.59703826904297, 'eval_loss': 17.544780731201172, 'bpb': 25.311768154388172}, {'step': 10.0, 'train_loss': 15.575966835021973, 'eval_loss': 15.416340827941895, 'bpb': 22.241078461125827}]


## Validation belongs inside the run


In [3]:
metrics = evaluate_model(model, tokenizer, text)
print(metrics)


{'loss': 14.698288917541504, 'bpb': 21.205148530890337}


## Batch size and accumulation change effective tokens per update


In [4]:
seq_len = model.config.block_size
for micro_batch, accumulation in [(4, 1), (4, 4), (8, 2)]:
    tokens_per_update = micro_batch * accumulation * seq_len
    print({'micro_batch': micro_batch, 'accumulation': accumulation, 'tokens_per_update': tokens_per_update})


{'micro_batch': 4, 'accumulation': 1, 'tokens_per_update': 128}
{'micro_batch': 4, 'accumulation': 4, 'tokens_per_update': 512}
{'micro_batch': 8, 'accumulation': 2, 'tokens_per_update': 512}


## Checkpoints and reports are part of the loop


In [5]:
out_dir = default_artifact_dir() / "training_loop_notebook"
checkpoint = save_checkpoint(
    out_dir / "tiny_checkpoint.pt",
    model=model,
    tokenizer=tokenizer,
    config=model.config,
    metadata={"lecture": LECTURE_NOTE_TITLE, "steps": 10},
    history=history,
)
report = write_json(
    out_dir / "train_report.json",
    {
        "checkpoint": str(checkpoint),
        "latest_metrics": metrics,
        "history": history,
        "config": model.config.__dict__,
    },
)
print("checkpoint:", checkpoint)
print("report    :", report)


checkpoint: /Users/montekkundan/Developer/ML/llm/artifacts/training_loop_notebook/tiny_checkpoint.pt
report    : /Users/montekkundan/Developer/ML/llm/artifacts/training_loop_notebook/train_report.json


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/gpt_train.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/scripts/base_train.py
- https://github.com/karpathy/nanochat/blob/master/nanochat/checkpoint_manager.py
